# 03 - Chains and Routing

Covers: composing runnables with the `|` operator (LCEL), multi-step chains, and routing to different chains based on an intermediate result.

In [ ]:
from llm_engineering import get_llm

llm = get_llm()

## Shared prompts and sample inputs

All examples below classify customer feedback. Swap `user_feedback` to try the other cases.

In [ ]:
from langchain_core.prompts import PromptTemplate

parse_template = PromptTemplate.from_template(
    "Parse and clean the following customer feedback for key information:\n\n{raw_feedback}"
)

summary_template = PromptTemplate.from_template(
    "Summarize this customer feedback in one concise sentence:\n\n{parsed_feedback}"
)

sentiment_template = PromptTemplate.from_template(
    "Determine the sentiment of this feedback and reply in one word as either "
    "'Positive', 'Neutral', or 'Negative':\n\n{feedback}"
)

FEEDBACK_NEUTRAL = (
    "The delivery was late, and the product was damaged when it arrived. "
    "However, the customer support team was very helpful in resolving the issue quickly."
)
FEEDBACK_POSITIVE = (
    "The customer service was fantastic. The representative was friendly, "
    "knowledgeable, and resolved my issue quickly."
)
FEEDBACK_NEGATIVE = (
    "I was extremely disappointed with the customer service. "
    "The representative was unhelpful and rude."
)

user_feedback = FEEDBACK_NEUTRAL

## A simple chain

`prompt | llm | parser` - the output of each step is the input of the next.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

sentiment_chain = sentiment_template | llm | StrOutputParser()
sentiment_chain.invoke({"feedback": user_feedback})

## A sequence of chains

`RunnableLambda` wraps a plain function so it can sit inside a chain. Here it converts the model's message into the dict that the next prompt expects.

In [ ]:
from langchain_core.runnables import RunnableLambda

to_parsed_feedback = RunnableLambda(lambda message: {"parsed_feedback": message.content})
to_feedback = RunnableLambda(lambda message: {"feedback": message.content})

pipeline = (
    parse_template | llm | to_parsed_feedback
    | summary_template | llm | to_feedback
    | sentiment_template | llm | StrOutputParser()
)

pipeline.invoke({"raw_feedback": user_feedback})

## Routing

Summarize the feedback, classify its sentiment, then pick a follow-up chain depending on the sentiment.

In [ ]:
thankyou_template = PromptTemplate.from_template(
    "Given the feedback, draft a thank you message for the user and request them "
    "to leave a positive rating on our webpage:\n\n{feedback}"
)
details_template = PromptTemplate.from_template(
    "Given the feedback, draft a message for the user and request them "
    "to provide more details about their concern:\n\n{feedback}"
)
apology_template = PromptTemplate.from_template(
    "Given the feedback, draft an apology message for the user and mention "
    "that their concern has been forwarded to the relevant department:\n\n{feedback}"
)

thankyou_chain = thankyou_template | llm | StrOutputParser()
details_chain = details_template | llm | StrOutputParser()
apology_chain = apology_template | llm | StrOutputParser()


def route(info: dict):
    """Return the follow-up chain that matches the sentiment."""
    sentiment = info["sentiment"].lower()
    if "positive" in sentiment:
        return thankyou_chain
    if "negative" in sentiment:
        return apology_chain
    return details_chain

In [ ]:
summary_chain = parse_template | llm | to_parsed_feedback | summary_template | llm | StrOutputParser()

summary = summary_chain.invoke({"raw_feedback": user_feedback})
sentiment = sentiment_chain.invoke({"feedback": summary})

print("Summary:", summary)
print("Sentiment:", sentiment)

full_chain = RunnableLambda(route)
full_chain.invoke({"feedback": summary, "sentiment": sentiment})